# YouTube Tech Review — NLP Processing Pipeline

## Overview
| Section | Stage | Output |
|---------|-------|--------|
| 0 | Dependency Installation | — |
| 1 | Imports & Config | — |
| 2 | MongoDB Connection | Collection handles |
| 3 | Data Cleaning | Cleaned comments DataFrame |
| 4 | Language Detection & Translation | `translated_comment` column |
| 5 | Tokenisation & Stopword Removal | `cleaned_tokens`, `cleaned_text` columns |
| 6 | Topic Modelling (BERTopic) | `dominant_topic`, `topic_label` columns |
| 7 | Sentiment Analysis (RoBERTa) | `sentiment` column |
| 8 | Product Name Mapping | `product_name` column |
| 9 | Final Assembly & Save | `youtube_final_data` in Cluster 2 |

## Setup
1. Copy `config.template.py` → `config.py` and fill in your credentials
2. Run **Section 0** once to install all dependencies
3. Run sections **1 → 9** in order

## Requirements
- Python 3.10+
- MongoDB Atlas (2 clusters — free tier 512MB limit applies)
- GPU recommended for Sections 6 & 7 (CPU works but is slow)

## Storage Notes
- **Cluster 1** — intermediate data (~500MB, free tier fills up)
- **Cluster 2** — final clean dataset only (`youtube_final_data`, ~50MB)
- If Cluster 1 gets full during Section 4, drop `youtube_translated_comments_main` before running Section 9

## Section 0 — Dependency Installation
Run **once** to bootstrap the environment. Installs all required packages and downloads the spaCy model.

In [ ]:
%pip install --upgrade transformers

In [ ]:
import subprocess, sys

packages = [
    'pymongo','transformers', 'sentencepiece', 'argostranslate', 'emoji', 'langdetect',
    'spacy', 'bertopic', 'matplotlib', 'plotly',
    'sentence-transformers', 'umap-learn', 'hdbscan',
    'scikit-learn', 'torch', 'transformers', 'tqdm', 'rapidfuzz',
]
subprocess.run([sys.executable, '-m', 'pip', 'install', *packages])
subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm'])
print('✅ All packages installed.')

## Section 1 — Imports & Config
Imports all third-party libraries and loads project configuration (credentials, model names, collection names, product catalogue) from `config.py`.

In [ ]:
# ── Standard Library ──────────────────────────────────────────────────────────
import sys, os, re, time

# ── Data ──────────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
from collections import Counter

# ── MongoDB ───────────────────────────────────────────────────────────────────
from pymongo import MongoClient

# ── HTTP ──────────────────────────────────────────────────────────────────────
import requests

# ── Text Cleaning ─────────────────────────────────────────────────────────────
import emoji

# ── Language Detection ────────────────────────────────────────────────────────
from langdetect import detect

# ── NLP ───────────────────────────────────────────────────────────────────────
import spacy

# ── Translation ─────────────────────────────────────────────────────────────
from transformers import pipeline
import argostranslate.package, argostranslate.translate

# ── Topic Modelling ───────────────────────────────────────────────────────────
from bertopic import BERTopic
from bertopic.representation import KeyBERTInspired
from umap import UMAP
from hdbscan import HDBSCAN
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ── Sentiment ─────────────────────────────────────────────────────────────────
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer as SentTokenizer

# ── Product Mapping ───────────────────────────────────────────────────────────
from rapidfuzz import process, fuzz

# ── Visualisation ────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import plotly.io as pio
from tqdm import tqdm
tqdm.pandas()

# ── Project Config ────────────────────────────────────────────────────────────
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))
from config import (
    # MongoDB
    MONGO_URI_CLUSTER1, MONGO_DB_CLUSTER1,
    MONGO_URI_CLUSTER2, MONGO_DB_CLUSTER2,
    # Collections
    COLLECTION_VIDEOS, COLLECTION_COMMENTS, COLLECTION_TRANSLATED,
    COLLECTION_TOPIC, COLLECTION_SENTIMENT, COLLECTION_FINAL,
    # APIs
    MICROSOFT_API_KEY, MICROSOFT_REGION, MICROSOFT_ENDPOINT,
    MICROSOFT_API_KEY_2, MICROSOFT_REGION_2, MICROSOFT_ENDPOINT_2,
    # Models
    SENTIMENT_MODEL, SPACY_MODEL,
    # Data
    PRODUCT_NAMES, BAD_TOPIC_LABELS,
)

print('✅ All libraries imported successfully.')

## Section 2 — MongoDB Connection
Establishes connections to both MongoDB clusters and creates collection handles.
- **Cluster 1** — raw and intermediate data (processing cluster)
- **Cluster 2** — final production data (RAG cluster)

In [ ]:
def get_client(uri: str, timeout_ms: int = 60_000) -> MongoClient:
    """Return a MongoClient with sensible timeouts."""
    return MongoClient(
        uri,
        serverSelectionTimeoutMS=timeout_ms,
        connectTimeoutMS=timeout_ms,
        socketTimeoutMS=None,
    )

# Cluster 1 — processing / intermediate data
client1 = get_client(MONGO_URI_CLUSTER1)
db1     = client1[MONGO_DB_CLUSTER1]

# Cluster 2 — RAG / production data
client2 = get_client(MONGO_URI_CLUSTER2)
db2     = client2[MONGO_DB_CLUSTER2]

# Collection handles — Cluster 1
videos_col      = db1[COLLECTION_VIDEOS]
comments_col    = db1[COLLECTION_COMMENTS]
translated_col  = db1[COLLECTION_TRANSLATED]
topic_col       = db1[COLLECTION_TOPIC]
sentiment_col   = db1[COLLECTION_SENTIMENT]

# Collection handles — Cluster 2
final_col       = db2[COLLECTION_FINAL]

print('✅ Connected to Cluster 1 (tech_reviews)  :', db1.list_collection_names())
print('✅ Connected to Cluster 2 (youtube_rag)   :', db2.list_collection_names())

## Section 3 — Data Cleaning
Loads raw comments from MongoDB and applies the following cleaning steps:
1. Removes URL-only spam (fewer than 4 real words after URL removal)
2. Strips URLs, emojis, non-ASCII characters, and punctuation
3. Normalises text to lowercase
4. Drops rows that are empty after cleaning
5. Removes generic spam and self-promotion phrases

In [ ]:
URL_PATTERN = re.compile(
    r'(https?://\S+|www\.\S+|\b\w+\.(com|net|org|io|co|ly|me|tv|gg)\S*)',
    re.IGNORECASE,
)

SPAM_KEYWORDS = [
    # Self-promotion
    'subscribe', 'check my channel', 'click the link', 'visit my channel',
    'check out my', 'sub to me', 'sub4sub', 'follow me', 'follow back',
    'i just posted', 'new video', 'watch my', 'link in bio',
    # Generic praise
    'great video', 'amazing content', 'nice video', 'good video',
    'best video', 'love this video', 'great review', 'nice review',
    'good review', 'best review', 'perfect video', 'excellent video',
    'wonderful video', 'fantastic video', 'awesome video', 'cool video',
    'nice one', 'well done', 'good job', 'great job', 'keep it up',
    'keep up the good work', 'great work', 'nice work', 'good work',
    'outstanding video', 'superb video', 'brilliant video',
]
SPAM_PATTERN = '|'.join(re.escape(k) for k in SPAM_KEYWORDS)


def clean_text(text: str) -> str:
    """Strip URLs, emojis, non-ASCII, punctuation, and normalise to lowercase."""
    text = str(text)
    text = URL_PATTERN.sub('', text)                  # 1. remove URLs
    text = emoji.demojize(text)                        # 2. emoji → :code:
    text = re.sub(r':[a-zA-Z_]+:', '', text)           # 3. remove :codes:
    text = text.encode('ascii', 'ignore').decode()     # 4. drop non-ASCII
    text = re.sub(r'[^\w\s]', '', text)               # 5. strip punctuation
    return text.lower().strip()


def is_url_only_spam(text: str) -> bool:
    """True only when < 4 real words remain after URL removal."""
    return len(URL_PATTERN.sub('', str(text)).strip().split()) < 4


# ── Load ──────────────────────────────────────────────────────────────────────
print('⏳ Loading comments from MongoDB...')
raw = list(tqdm(
    comments_col.find({}, {'_id': 0, 'video_id': 1, 'product': 1, 'comment': 1}),
    desc='Fetching',
))
df = pd.DataFrame(raw)
print(f'✅ Loaded {len(df):,} comments. Null counts:\n{df.isnull().sum()}')

# ── URL-only spam ─────────────────────────────────────────────────────────────
before = len(df)
df = df[~df['comment'].apply(is_url_only_spam)].reset_index(drop=True)
print(f'🔗 Removed {before - len(df):,} URL-only spam. Remaining: {len(df):,}')

# ── Clean text ────────────────────────────────────────────────────────────────
print('⏳ Cleaning text...')
df['comment'] = df['comment'].progress_apply(clean_text)
df['product'] = df['product'].progress_apply(clean_text)

before = len(df)
df = df[df['comment'].str.strip().str.len() > 0].reset_index(drop=True)
print(f'Dropped {before - len(df):,} empty rows after cleaning.')

# ── Spam keywords ─────────────────────────────────────────────────────────────
before = len(df)
df = df[~df['comment'].str.contains(SPAM_PATTERN, case=False, na=False)].reset_index(drop=True)
print(f'✅ Removed {before - len(df):,} spam comments. Remaining: {len(df):,}')
df.head()

## Section 4 — Language Detection & Translation
Detects the language of every comment, re-detects borderline cases, translates non-English comments to English using Helsinki-NLP MarianMT with Argos Translate as a fallback, then saves the result to Cluster 2.

### 4.1 — Load Helsinki-NLP Translation Model
Loads the `Helsinki-NLP/opus-mt-mul-en` multilingual-to-English model onto the available device (GPU if available, otherwise CPU).

In [ ]:
from transformers import MarianTokenizer, MarianMTModel
import argostranslate.package, argostranslate.translate
import torch

print("⏳ Loading Helsinki-NLP translation model...")
model_name = "Helsinki-NLP/opus-mt-mul-en"
tokenizer  = MarianTokenizer.from_pretrained(model_name)
model      = MarianMTModel.from_pretrained(model_name)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device)
model.eval()
print(f"✅ Helsinki-NLP loaded on: {device}")

### 4.2 — Define Helper Functions
Defines `safe_detect` (language detection with error handling), `helsinki_translate` (batch translation using the MarianMT model), and `translate_batch` (orchestrates Helsinki + Argos fallback).

In [ ]:
def safe_detect(text: str) -> str:
    try:
        return detect(str(text))
    except Exception:
        return 'unknown'


def helsinki_translate(texts: list) -> list:
    inputs = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512,
    ).to(device)

    with torch.no_grad():
        translated = model.generate(**inputs)

    return [tokenizer.decode(t, skip_special_tokens=True) for t in translated]


def translate_batch(texts: list, langs: list, batch_size: int = 32) -> list:
    results = []

    # ── Pass 1: Helsinki ──────────────────────────────────────────────────────
    print(f"⏳ Helsinki translating {len(texts):,} comments in batches of {batch_size}...")
    for i in tqdm(range(0, len(texts), batch_size), desc="Helsinki", unit="batch"):
        batch = texts[i: i + batch_size]
        try:
            results.extend(helsinki_translate(batch))
        except Exception as e:
            print(f"  ⚠️ Helsinki failed on batch {i//batch_size}: {e}")
            results.extend([None] * len(batch))

    # ── Pass 2: Argos fallback ────────────────────────────────────────────────
    failed_indices = [i for i, r in enumerate(results) if not r]
    if failed_indices:
        print(f"\n⏳ Argos fallback for {len(failed_indices):,} failed comments...")

        argos_langs = list(set(langs[i] for i in failed_indices))
        print(f"   Languages needing Argos: {argos_langs}")
        argostranslate.package.update_package_index()
        available = argostranslate.package.get_available_packages()
        for lang_code in argos_langs:
            pkg = next(
                (p for p in available if p.from_code == lang_code and p.to_code == 'en'),
                None,
            )
            if pkg:
                argostranslate.package.install_from_path(pkg.download())
                print(f"  ✅ Installed: {lang_code}→en")
            else:
                print(f"  ⚠️ No Argos package for: {lang_code}")

        for i in tqdm(failed_indices, desc="Argos fallback", unit="comment"):
            try:
                results[i] = argostranslate.translate.translate(texts[i], langs[i], 'en')
            except Exception:
                results[i] = texts[i]  # keep original as last resort
    else:
        print("✅ No Argos fallback needed — Helsinki handled everything.")

    return results

### 4.3 — Detect Languages & Drop Unknown Rows
Runs language detection on every comment using `langdetect` and removes rows where the language could not be identified.

In [ ]:
print("⏳ Detecting languages...")
df['language'] = df['comment'].progress_apply(safe_detect)
print("Language distribution:", dict(Counter(df['language'])))

before = len(df)
df = df[df['language'] != 'unknown'].reset_index(drop=True)
print(f"Dropped {before - len(df):,} unknown-language rows. Remaining: {len(df):,}")


### 4.4 — Split English vs Non-English & Re-detect
Separates English from non-English comments, then re-detects the non-English subset to catch any mis-labelled entries before translation.

In [ ]:
df['translated_comment'] = df['comment']
english_df     = df[df['language'] == 'en'].copy()
non_english_df = df[df['language'] != 'en'].copy()
print(f"English: {len(english_df):,}  |  Non-English: {len(non_english_df):,}")

print("\n⏳ Re-detecting non-English rows to catch mis-labelled entries...")
non_english_df['re_detected'] = non_english_df['comment'].progress_apply(safe_detect)

actually_english = non_english_df[non_english_df['re_detected'] == 'en'].copy()
truly_foreign    = non_english_df[non_english_df['re_detected'] != 'en'].copy()

actually_english['language'] = 'en'
actually_english['translated_comment'] = actually_english['comment']

print(f"Fixed mis-detected : {len(actually_english):,}")
print(f"Truly foreign      : {len(truly_foreign):,}  → these will be translated")


### 4.5 — Translate Non-English Comments
Runs the truly non-English comments through `translate_batch` (Helsinki → Argos fallback) to produce English translations.

In [ ]:
print(f"⏳ Starting translation for {len(truly_foreign):,} non-English comments...\n")

truly_foreign = truly_foreign.copy()
truly_foreign['translated_comment'] = translate_batch(
    truly_foreign['comment'].tolist(),
    truly_foreign['re_detected'].tolist(),
)

print(f"\n✅ Translation complete. Sample output:")
print(truly_foreign[['comment', 'translated_comment']].head(3).to_string())


### 4.6 — Merge All Parts & Remove Duplicates
Concatenates the English, re-labelled, and translated subsets back into a single DataFrame and drops any duplicate rows.

In [ ]:
print("⏳ Merging all parts...")
final_df = pd.concat([english_df, actually_english, truly_foreign], ignore_index=True)

before = len(final_df)
final_df.drop_duplicates(subset=['video_id', 'comment'], inplace=True)
print(f"Removed {before - len(final_df):,} duplicates. Remaining: {len(final_df):,}")

### 4.7 — Final Language Validation
Runs one final language check on the translated comments and drops any rows that are still not in English after translation.

In [ ]:
print("⏳ Final language check on translated comments...")
before = len(final_df)

final_df['_detected_trans'] = final_df['translated_comment'].progress_apply(safe_detect)
final_df = final_df[
    (final_df['language'] == 'en') | (final_df['_detected_trans'] == 'en')
].reset_index(drop=True)
final_df.drop(columns=['_detected_trans'], inplace=True)

print(f"Dropped {before - len(final_df):,} remaining non-English rows.")
print(f"Final count: {len(final_df):,} rows ready to save.")


### 4.8 — Save Translated Data to Cluster 2
Persists the fully translated DataFrame to the `youtube_translated_comments_main` collection in Cluster 2 in batches of 5,000.

In [ ]:
print(f"⏳ Saving {len(final_df):,} records to '{COLLECTION_TRANSLATED}' in Cluster 2...")

client2      = MongoClient(MONGO_URI_CLUSTER2)
db2          = client2[MONGO_DB_CLUSTER2]
translated_col2 = db2[COLLECTION_TRANSLATED]

translated_col2.delete_many({})

records = final_df.to_dict(orient='records')
for i in tqdm(range(0, len(records), 5000), desc="Saving to MongoDB", unit="batch"):
    translated_col2.insert_many(records[i: i + 5000])

verified = translated_col2.count_documents({})
print(f"✅ Verified: {verified:,} docs saved to '{COLLECTION_TRANSLATED}' in Cluster 2.")
df = final_df.copy()

## Section 5 — Tokenisation & Stopword Removal
Loads the translated data from Cluster 2, then uses spaCy (`en_core_web_sm`) to tokenise, lemmatise, and remove stopwords from every comment.

> **Note:** `cleaned_tokens` / `cleaned_text` are bag-of-words representations used for exploratory analysis. BERTopic in Section 6 uses the raw `translated_comment` (natural sentences) for better semantic embedding.

In [ ]:
print('⏳ Loading from MongoDB (translated collection)...')
client2         = MongoClient(MONGO_URI_CLUSTER2)
db2             = client2[MONGO_DB_CLUSTER2]
translated_col2 = db2[COLLECTION_TRANSLATED]
df = pd.DataFrame(list(translated_col2.find())).drop(columns=['_id'], errors='ignore')
print(f'✅ Loaded {len(df):,} rows.')

print('⏳ Loading spaCy model...')
nlp        = spacy.load(SPACY_MODEL)
STOP_WORDS = nlp.Defaults.stop_words
print('✅ spaCy loaded.')


def spacy_tokenize(text: str) -> list:
    """Tokenise, remove stopwords, and lemmatise."""
    doc = nlp(str(text))
    return [
        token.lemma_
        for token in doc
        if token.text.lower() not in STOP_WORDS
        and not token.is_punct
        and not token.is_space
        and len(token.text) > 1
    ]


print('⏳ Tokenising...')
df['cleaned_tokens'] = df['translated_comment'].progress_apply(spacy_tokenize)
df['cleaned_text']   = df['cleaned_tokens'].apply(' '.join)

before = len(df)
df = df[df['translated_comment'].str.strip().str.len() > 5].reset_index(drop=True)
print(f'✅ Done. Dropped {before - len(df):,} short/empty rows.')
df[['translated_comment', 'cleaned_tokens']].head(3)

## Section 6 — Topic Modelling (BERTopic)
Clusters comments into topics using BERTopic with the following pipeline:
1. **Embeddings** — `all-MiniLM-L6-v2` sentence transformer
2. **Dimensionality reduction** — UMAP (`n_neighbors=10`, `n_components=5`)
3. **Clustering** — HDBSCAN (`min_cluster_size=30`, `min_samples=5`)
4. **Representation** — KeyBERTInspired top keywords
5. **Outlier reduction** — probability strategy (threshold 0.05) then cosine-similarity fallback

Only comments with 5 or more words are passed to BERTopic to reduce noise.

### 6.1 — Run BERTopic & Assign Topic Labels
Fits the BERTopic model, reduces outliers, and maps each comment to a human-readable topic label (top 3 keywords).

In [ ]:
df = df[df['translated_comment'].str.split().str.len() >= 5].reset_index(drop=True)
docs = df['translated_comment'].fillna('').tolist()
print(f'⏳ Preparing {len(docs):,} documents...')

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

umap_model = UMAP(
    n_neighbors=10,     
    n_components=5,
    min_dist=0.0,
    metric='cosine',
    random_state=42,
)
hdbscan_model = HDBSCAN(
    min_cluster_size=30,   
    min_samples=5,         
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True,
)
topic_model = BERTopic(
    embedding_model=embedding_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    representation_model=KeyBERTInspired(),
    nr_topics='auto',
    top_n_words=10,
    verbose=True,
)

print('⏳ Running BERTopic fit_transform (may take 15–25 min on 200k+ rows)...')
topics, probs = topic_model.fit_transform(docs)
topics = list(topics)
print(f'Before outlier reduction — Uncategorised: {sum(t == -1 for t in topics):,}')

# ── Reduce outliers (probability strategy) ────────────────────────────────────
print('⏳ Reducing outliers...')
topics = list(topic_model.reduce_outliers(docs, topics, probabilities=probs,
                                           strategy='probabilities', threshold=0.05))
topic_model.update_topics(docs, topics=topics)
print(f'After outlier reduction  — Uncategorised: {sum(t == -1 for t in topics):,}')

# ── Cosine similarity fallback ────────────────────────────────────────────────
remaining_idx = [i for i, t in enumerate(topics) if t == -1]
if remaining_idx:
    print(f'⏳ Cosine fallback for {len(remaining_idx):,} remaining outliers...')
    embeddings    = embedding_model.encode(docs, show_progress_bar=True, batch_size=256)
    topic_ids     = sorted(set(topics) - {-1})
    centroids     = np.array([
        embeddings[[i for i, t in enumerate(topics) if t == tid]].mean(axis=0)
        for tid in topic_ids
    ])
    sims          = cosine_similarity(embeddings[remaining_idx], centroids)
    for idx, row in zip(remaining_idx, sims):
        topics[idx] = topic_ids[int(np.argmax(row))]
    print('✅ All remaining outliers reassigned.')

# ── Build readable labels ─────────────────────────────────────────────────────
topic_info   = topic_model.get_topic_info()
topic_labels = {
    tid: ('Uncategorized' if tid == -1
          else ', '.join(w[0] for w in (topic_model.get_topic(tid) or [])[:3]) or 'Uncategorized')
    for tid in topic_info['Topic'].unique()
}
df['dominant_topic'] = topics
df['topic_label']    = df['dominant_topic'].map(topic_labels)

categorized_pct = (df['dominant_topic'] != -1).mean() * 100
print(f'\n📊 Topics: {len(topic_labels)-1}  |  Categorised: {categorized_pct:.1f}%')
df[['translated_comment', 'dominant_topic', 'topic_label']].head()

### 6.2 — Visualise Topics & Save to Cluster 2
Plots the top 15 topics by comment count and persists the topic-labelled DataFrame to `youtube_topic_model` in Cluster 2.

In [ ]:
# ── Visualise ─────────────────────────────────────────────────────────────────
topic_counts = df['topic_label'].value_counts().reset_index()
topic_counts.columns = ['Topic', 'Count']
plt.figure(figsize=(12, 7))
plt.barh(topic_counts['Topic'][:15], topic_counts['Count'][:15], color='steelblue')
plt.xlabel('Number of Comments')
plt.title('Top 15 Topics (BERTopic)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

# ── Persist ───────────────────────────────────────────────────────────────────
client2   = MongoClient(MONGO_URI_CLUSTER2)
db2       = client2[MONGO_DB_CLUSTER2]
topic_col2 = db2[COLLECTION_TOPIC]

topic_col2.drop()
records = df.to_dict(orient='records')
print(f'⏳ Saving {len(records):,} topic records to Cluster 2...')
for i in tqdm(range(0, len(records), 5000), desc='Saving', unit='batch'):
    topic_col2.insert_many(records[i: i + 5000])
    time.sleep(0.3)
print(f'✅ Saved {len(records):,} records to {COLLECTION_TOPIC} in Cluster 2.')

## Section 7 — Sentiment Analysis (RoBERTa)
Loads topic-labelled data from Cluster 2 and classifies each comment as **Positive**, **Neutral**, or **Negative** using `cardiffnlp/twitter-roberta-base-sentiment` — a RoBERTa model fine-tuned on social-media text. Runs in batches of 64 with GPU auto-detection. Results are saved to `youtube_Sentiment_Analysis` in Cluster 2.

In [ ]:
SENTIMENT_LABELS = ['Negative', 'Neutral', 'Positive']

print('⏳ Loading sentiment model...')
sent_tokenizer = SentTokenizer.from_pretrained(SENTIMENT_MODEL)
sent_model     = AutoModelForSequenceClassification.from_pretrained(SENTIMENT_MODEL)
DEVICE         = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sent_model     = sent_model.to(DEVICE).eval()
print(f'✅ Model loaded on: {DEVICE}')


def get_sentiment_batch(texts: list, batch_size: int = 64) -> list:
    """Classify a list of texts; returns a list of label strings."""
    results = []
    for start in tqdm(range(0, len(texts), batch_size), desc='Sentiment batches'):
        batch  = texts[start: start + batch_size]
        inputs = sent_tokenizer(
            batch, return_tensors='pt',
            truncation=True, padding=True, max_length=128,
        )
        inputs = {k: v.to(DEVICE) for k, v in inputs.items()}
        with torch.no_grad():
            preds = torch.argmax(sent_model(**inputs).logits, dim=1).cpu().numpy()
        results.extend(SENTIMENT_LABELS[p] for p in preds)
    return results


# ── Load topic data ───────────────────────────────────────────────────────────
print('⏳ Loading topic data from MongoDB...')
client2    = MongoClient(MONGO_URI_CLUSTER2)
db2        = client2[MONGO_DB_CLUSTER2]
topic_col2 = db2[COLLECTION_TOPIC]
df = pd.DataFrame(list(topic_col2.find())).drop(columns=['_id'], errors='ignore')
print(f'✅ Loaded {len(df):,} rows.')

# ── Run sentiment ─────────────────────────────────────────────────────────────
print('⏳ Running batch sentiment analysis...')
df['sentiment'] = get_sentiment_batch(df['translated_comment'].fillna('').tolist())
print('✅ Sentiment analysis complete.')
print(df['sentiment'].value_counts().to_string())

# ── Persist to Cluster 2 ──────────────────────────────────────────────────────
sent_col_2 = db2[COLLECTION_SENTIMENT]
sent_col_2.drop()
records = df.to_dict(orient='records')
print(f'⏳ Saving {len(records):,} records to Cluster 2...')
for i in tqdm(range(0, len(records), 5000), desc='Saving', unit='batch'):
    sent_col_2.insert_many(records[i: i + 5000])
    time.sleep(0.5)
print(f'✅ Saved {len(records):,} records to {COLLECTION_SENTIMENT} in Cluster 2.')

## Section 8 — Product Name Mapping
Normalises the raw `product` field scraped from YouTube to a canonical product name from the `PRODUCT_NAMES` list in `config.py` using:
1. **Exact substring match** — fast and reliable for clean product strings
2. **RapidFuzz fuzzy match** — `token_sort_ratio` with a threshold of ≥ 85 for slightly noisy strings

Rows that match neither method are labelled `Unknown`.

In [ ]:
def map_product_name(raw_product: str) -> str:
    """Return the canonical product name or 'Unknown'."""
    lower = str(raw_product).lower()
    for product in PRODUCT_NAMES:
        if product in lower:
            return product
    result = process.extractOne(lower, PRODUCT_NAMES, scorer=fuzz.token_sort_ratio)
    if result:
        best, score, _ = result
        return best if score >= 85 else 'Unknown'
    return 'Unknown'


print('⏳ Mapping product names...')
df['product_name'] = df['product'].progress_apply(map_product_name)
print('✅ Product mapping complete.')

unknown_count = (df['product_name'] == 'Unknown').sum()
print(f'Unknown: {unknown_count:,} / {len(df):,}')
if unknown_count > 0:
    print('\n🔍 Top unknowns (expand PRODUCT_NAMES in config.py if needed):')
    print(df[df['product_name'] == 'Unknown']['product'].value_counts().head(20).to_string())
df.head()

## Section 9 — Final Assembly & Save
Filters out junk topic labels, selects the final columns, and saves the clean production-ready dataset to `youtube_final_data` in Cluster 2.

**Final columns:** `video_id`, `translated_comment`, `sentiment`, `topic_label`, `product_name`

### 9.1 — Free Up Space (Drop Translated Collection)
Drops the large intermediate `youtube_translated_comments_main` collection from Cluster 2 to free up storage before the final save.

In [ ]:
client2 = MongoClient(MONGO_URI_CLUSTER2)
db2 = client2[MONGO_DB_CLUSTER2]
db2['youtube_translated_comments_main'].drop()
print("✅ Dropped.")

### 9.2 — Reload Config & Inspect Topic Distribution
Reloads `config.py` to pick up any updated `BAD_TOPIC_LABELS`, then prints the top 20 topic labels by comment count for review.

In [ ]:
import importlib
import config
importlib.reload(config)
from config import *
print(BAD_TOPIC_LABELS)

In [ ]:
print(df['topic_label'].value_counts().head(20).to_string())

### 9.3 — Drop Junk Topic Labels
Filters out rows whose `topic_label` is in `BAD_TOPIC_LABELS` (stopword clusters, spam topics, noise).

In [ ]:
before = len(df)
df = df[~df['topic_label'].isin(BAD_TOPIC_LABELS)].reset_index(drop=True)
print(f'Dropped {before - len(df):,} rows with junk topic labels.')

### 9.4 — Save Final Dataset to Cluster 2
Selects the 5 final columns, validates they exist, then saves the clean records to `youtube_final_data` in Cluster 2 in batches of 5,000.

In [ ]:
KEEP_COLUMNS = ['video_id', 'translated_comment', 'sentiment', 'topic_label', 'product_name']

# ── Validate columns ──────────────────────────────────────────────────────────
missing = [c for c in KEEP_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f'Missing columns before save: {missing}')

df_final = df[KEEP_COLUMNS].copy()

# ── Save to Cluster 2 ─────────────────────────────────────────────────────────
client2   = MongoClient(MONGO_URI_CLUSTER2)
db2       = client2[MONGO_DB_CLUSTER2]
final_col = db2[COLLECTION_FINAL]
records = df_final.to_dict(orient='records')
print(f'⏳ Saving {len(records):,} records to Cluster 2 → "{COLLECTION_FINAL}"...')
final_col.drop()
for i in tqdm(range(0, len(records), 5000), desc='Saving', unit='batch'):
    final_col.insert_many(records[i: i + 5000])
    time.sleep(0.3)

verified = final_col.count_documents({})
print(f'\n✅ Verified: {verified:,} docs in "{COLLECTION_FINAL}"')

### 9.5 — (Optional) Re-filter & Inspect Topics
Optional cells for re-applying the junk filter after updating `BAD_TOPIC_LABELS`, inspecting the remaining topic distribution, and spot-checking sample comments per topic.

In [ ]:
importlib.reload(config)
from config import *

before = len(df)
df = df[~df['topic_label'].isin(BAD_TOPIC_LABELS)].reset_index(drop=True)
print(f'Dropped {before - len(df):,} rows. Remaining: {len(df):,}')

In [ ]:
print(df['topic_label'].value_counts().head(20).to_string())

In [ ]:
for topic in df['topic_label'].unique():
    print(f"\n{'='*60}")
    print(f"Topic: {topic}")
    print('='*60)
    samples = df[df['topic_label'] == topic]['translated_comment'].head(10).tolist()
    for i, comment in enumerate(samples, 1):
        print(f"{i}. {comment}")

In [ ]:
print(df[df['topic_label'] == 'zone, school, fast']['translated_comment'].head(10).tolist())